# Chapter 4 — Trees and Graphs

**Vocabulary**: binary tree, binary *search* tree (left ≤ node < right), balanced, complete, full, perfect.
**Traversals**: in-order, pre-order, post-order.
**Binary heaps** (min-heap): `heapq`. **Tries**: prefix trees.
**Graphs**: adjacency lists; **DFS** (recursion/stack) and **BFS** (queue); **bidirectional search**.

Core implementations first, then problems 4.1–4.12.

In [1]:
from __future__ import annotations

import heapq
import random
from collections import defaultdict, deque
from collections.abc import Iterator
from dataclasses import dataclass, field


@dataclass(eq=False)
class TreeNode:
    val: int
    left: TreeNode | None = None
    right: TreeNode | None = None
    parent: TreeNode | None = field(default=None, repr=False)


def in_order(n: TreeNode | None) -> Iterator[int]:
    if n:
        yield from in_order(n.left)
        yield n.val
        yield from in_order(n.right)


def pre_order(n: TreeNode | None) -> Iterator[int]:
    if n:
        yield n.val
        yield from pre_order(n.left)
        yield from pre_order(n.right)


def post_order(n: TreeNode | None) -> Iterator[int]:
    if n:
        yield from post_order(n.left)
        yield from post_order(n.right)
        yield n.val


def bst_insert(root: TreeNode | None, val: int) -> TreeNode:
    if root is None:
        return TreeNode(val)
    if val <= root.val:
        root.left = bst_insert(root.left, val)
        root.left.parent = root
    else:
        root.right = bst_insert(root.right, val)
        root.right.parent = root
    return root


def build_bst(values: list[int]) -> TreeNode | None:
    root = None
    for v in values:
        root = bst_insert(root, v)
    return root


def find(root: TreeNode | None, val: int) -> TreeNode | None:
    while root and root.val != val:
        root = root.left if val < root.val else root.right
    return root


t = build_bst([8, 4, 10, 2, 6, 20])
assert list(in_order(t)) == [2, 4, 6, 8, 10, 20]
assert list(pre_order(t)) == [8, 4, 2, 6, 10, 20]
assert list(post_order(t)) == [2, 6, 4, 20, 10, 8]
print("traversals ok")

traversals ok


### Min-heap (hand-rolled, then `heapq`)

In [2]:
class MinHeap:
    def __init__(self) -> None:
        self.a: list[int] = []

    def insert(self, v: int) -> None:
        self.a.append(v)
        i = len(self.a) - 1
        while i > 0 and self.a[(i - 1) // 2] > self.a[i]:  # bubble up
            p = (i - 1) // 2
            self.a[i], self.a[p] = self.a[p], self.a[i]
            i = p

    def extract_min(self) -> int:
        a = self.a
        top, last = a[0], a.pop()
        if a:
            a[0] = last
            i = 0
            while True:  # bubble down
                l, r, smallest = 2 * i + 1, 2 * i + 2, i
                if l < len(a) and a[l] < a[smallest]: smallest = l
                if r < len(a) and a[r] < a[smallest]: smallest = r
                if smallest == i: break
                a[i], a[smallest] = a[smallest], a[i]
                i = smallest
        return top


h = MinHeap()
data = random.sample(range(100), 20)
for v in data:
    h.insert(v)
assert [h.extract_min() for _ in range(20)] == sorted(data)
assert heapq.nsmallest(3, data) == sorted(data)[:3]
print("heap ok")

heap ok


### Trie

In [3]:
class Trie:
    def __init__(self, words: list[str] = ()) -> None:  # type: ignore[assignment]
        self.root: dict = {}
        for w in words:
            self.insert(w)

    def insert(self, word: str) -> None:
        node = self.root
        for c in word:
            node = node.setdefault(c, {})
        node["$"] = True  # terminator

    def _walk(self, prefix: str) -> dict | None:
        node = self.root
        for c in prefix:
            if c not in node:
                return None
            node = node[c]
        return node

    def contains(self, word: str) -> bool:
        node = self._walk(word)
        return bool(node and node.get("$"))

    def has_prefix(self, prefix: str) -> bool:
        return self._walk(prefix) is not None


trie = Trie(["many", "my", "lie", "a"])
assert trie.contains("my") and not trie.contains("ma") and trie.has_prefix("ma")
print("trie ok")

trie ok


### Graph + DFS/BFS

In [4]:
Graph = dict[str, list[str]]


def dfs(g: Graph, start: str) -> list[str]:
    visited: list[str] = []
    seen: set[str] = set()

    def visit(n: str) -> None:
        seen.add(n)
        visited.append(n)
        for nb in g.get(n, []):
            if nb not in seen:
                visit(nb)

    visit(start)
    return visited


def bfs(g: Graph, start: str) -> list[str]:
    order, seen, q = [], {start}, deque([start])
    while q:
        n = q.popleft()
        order.append(n)
        for nb in g.get(n, []):
            if nb not in seen:
                seen.add(nb)
                q.append(nb)
    return order


g: Graph = {"0": ["1", "4", "5"], "1": ["3", "4"], "2": ["1"], "3": ["2", "4"], "4": [], "5": []}
print("DFS:", dfs(g, "0"))
print("BFS:", bfs(g, "0"))
assert bfs(g, "0") == ["0", "1", "4", "5", "3", "2"]

DFS: ['0', '1', '3', '2', '4', '5']
BFS: ['0', '1', '4', '5', '3', '2']


## 4.1 Route Between Nodes
In a directed graph, is there a path from S to E? BFS (finds shortest path too).

In [5]:
def has_route(g: Graph, start: str, end: str) -> bool:
    if start == end:
        return True
    seen, q = {start}, deque([start])
    while q:
        for nb in g.get(q.popleft(), []):
            if nb == end:
                return True
            if nb not in seen:
                seen.add(nb)
                q.append(nb)
    return False


assert has_route(g, "0", "2") and not has_route(g, "4", "0")
print("4.1 ok")

4.1 ok


## 4.2 Minimal Tree
Given a sorted array of unique ints, build a BST of minimal height: recursively make the middle element the root.

In [6]:
def minimal_tree(xs: list[int], lo: int = 0, hi: int | None = None) -> TreeNode | None:
    if hi is None:
        hi = len(xs) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    node = TreeNode(xs[mid])
    node.left = minimal_tree(xs, lo, mid - 1)
    node.right = minimal_tree(xs, mid + 1, hi)
    return node


def height(n: TreeNode | None) -> int:
    return -1 if n is None else 1 + max(height(n.left), height(n.right))


mt = minimal_tree(list(range(15)))
assert list(in_order(mt)) == list(range(15)) and height(mt) == 3
print("4.2 ok")

4.2 ok


## 4.3 List of Depths
Produce one list per depth level of a binary tree (the book uses linked lists; Python lists are fine).

In [7]:
def list_of_depths(root: TreeNode | None) -> list[list[int]]:
    levels: list[list[int]] = []
    current = [root] if root else []
    while current:
        levels.append([n.val for n in current])
        current = [c for n in current for c in (n.left, n.right) if c]
    return levels


assert list_of_depths(minimal_tree([1, 2, 3, 4, 5, 6, 7])) == [[4], [2, 6], [1, 3, 5, 7]]
print("4.3 ok")

4.3 ok


## 4.4 Check Balanced
Balanced = for every node, subtree heights differ by ≤ 1. Return heights up the tree and use a sentinel to short-circuit on imbalance → `O(N)`.

In [8]:
UNBALANCED = -2


def check_height(n: TreeNode | None) -> int:
    if n is None:
        return -1
    lh = check_height(n.left)
    if lh == UNBALANCED:
        return UNBALANCED
    rh = check_height(n.right)
    if rh == UNBALANCED or abs(lh - rh) > 1:
        return UNBALANCED
    return 1 + max(lh, rh)


def is_balanced(root: TreeNode | None) -> bool:
    return check_height(root) != UNBALANCED


assert is_balanced(minimal_tree(list(range(10))))
assert not is_balanced(build_bst([1, 2, 3]))
print("4.4 ok")

4.4 ok


## 4.5 Validate BST
Pass down (min, max) bounds. Note: checking only `left < node < right` locally is a classic bug.

In [9]:
def is_bst(n: TreeNode | None, lo: float = float("-inf"), hi: float = float("inf")) -> bool:
    if n is None:
        return True
    # book convention: left <= node < right
    if not (lo < n.val <= hi):
        return False
    return is_bst(n.left, lo, n.val) and is_bst(n.right, n.val, hi)


bad = TreeNode(20, TreeNode(10, None, TreeNode(25)), TreeNode(30))  # 25 is in left subtree of 20
assert is_bst(build_bst([5, 3, 8, 3, 9])) and not is_bst(bad)
print("4.5 ok")

4.5 ok


## 4.6 Successor
Find the in-order successor of a node in a BST where each node has a parent link.
- If it has a right subtree → leftmost node of that subtree.
- Else walk up until we come from a left child.

In [10]:
def successor(n: TreeNode | None) -> TreeNode | None:
    if n is None:
        return None
    if n.right:
        n = n.right
        while n.left:
            n = n.left
        return n
    while n.parent and n.parent.right is n:
        n = n.parent
    return n.parent


root = build_bst([20, 8, 22, 4, 12, 10, 14])
assert successor(find(root, 8)).val == 10
assert successor(find(root, 14)).val == 20
assert successor(find(root, 22)) is None
print("4.6 ok")

4.6 ok


## 4.7 Build Order
Given projects and (dependency, project) pairs, produce an order where each project comes after its dependencies, or report a cycle. This is **topological sort** (Kahn's algorithm). Python 3.9+ also ships `graphlib.TopologicalSorter`.

In [11]:
from graphlib import CycleError, TopologicalSorter


def build_order(projects: list[str], deps: list[tuple[str, str]]) -> list[str]:
    incoming = {p: 0 for p in projects}
    edges: dict[str, list[str]] = defaultdict(list)
    for before, after in deps:
        edges[before].append(after)
        incoming[after] += 1
    ready = deque(p for p in projects if incoming[p] == 0)
    order: list[str] = []
    while ready:
        p = ready.popleft()
        order.append(p)
        for nxt in edges[p]:
            incoming[nxt] -= 1
            if incoming[nxt] == 0:
                ready.append(nxt)
    if len(order) != len(projects):
        raise ValueError("cycle detected - no valid build order")
    return order


def build_order_stdlib(projects: list[str], deps: list[tuple[str, str]]) -> list[str]:
    ts = TopologicalSorter({p: set() for p in projects})
    for before, after in deps:
        ts.add(after, before)
    try:
        return list(ts.static_order())
    except CycleError as e:
        raise ValueError("cycle detected") from e


projects = list("abcdef")
deps = [("a", "d"), ("f", "b"), ("b", "d"), ("f", "a"), ("d", "c")]
for fn in (build_order, build_order_stdlib):
    order = fn(projects, deps)
    assert all(order.index(x) < order.index(y) for x, y in deps)
    try:
        fn(["a", "b"], [("a", "b"), ("b", "a")])
        raise AssertionError("expected cycle")
    except ValueError:
        pass
print(build_order(projects, deps))

['e', 'f', 'b', 'a', 'd', 'c']


## 4.8 First Common Ancestor
Find the lowest common ancestor of two nodes in a binary tree (not necessarily a BST), without storing extra nodes. Recursive approach returns `(node, is_ancestor)`; the simpler common version below assumes both nodes are present, followed by a version that verifies presence.

In [12]:
def lca(root: TreeNode | None, p: TreeNode, q: TreeNode) -> TreeNode | None:
    if root is None or root is p or root is q:
        return root
    left = lca(root.left, p, q)
    right = lca(root.right, p, q)
    if left and right:
        return root
    return left or right


def lca_safe(root: TreeNode | None, p: TreeNode, q: TreeNode) -> TreeNode | None:
    """Returns None if either node isn't in the tree."""
    found = 0

    def helper(n: TreeNode | None) -> TreeNode | None:
        nonlocal found
        if n is None:
            return None
        left, right = helper(n.left), helper(n.right)
        if n is p or n is q:
            found += 1
            return n
        if left and right:
            return n
        return left or right

    result = helper(root)
    return result if found == 2 else None


root = build_bst([20, 8, 22, 4, 12, 10, 14])
n10, n14, n4, n22 = (find(root, v) for v in (10, 14, 4, 22))
assert lca(root, n10, n14).val == 12
assert lca(root, n4, n14).val == 8
assert lca_safe(root, n4, n22).val == 20
assert lca_safe(root, n4, TreeNode(99)) is None
print("4.8 ok")

4.8 ok


## 4.9 BST Sequences
A BST was built by inserting elements left-to-right from an array. Print every array that could have produced this tree. The root must come first; then any **weave** of the left-subtree sequences with the right-subtree sequences (preserving the order within each).

In [13]:
def weave(a: list[int], b: list[int], prefix: list[int], out: list[list[int]]) -> None:
    if not a or not b:
        out.append(prefix + a + b)
        return
    weave(a[1:], b, prefix + [a[0]], out)
    weave(a, b[1:], prefix + [b[0]], out)


def bst_sequences(n: TreeNode | None) -> list[list[int]]:
    if n is None:
        return [[]]
    result: list[list[int]] = []
    for left in bst_sequences(n.left):
        for right in bst_sequences(n.right):
            woven: list[list[int]] = []
            weave(left, right, [n.val], woven)
            result.extend(woven)
    return result


seqs = bst_sequences(build_bst([2, 1, 3]))
assert sorted(seqs) == [[2, 1, 3], [2, 3, 1]]
tree = build_bst([50, 20, 60, 10, 25, 70])
seqs = bst_sequences(tree)
assert all(list(pre_order(build_bst(s))) == list(pre_order(tree)) for s in seqs)
print(len(seqs), "sequences, e.g.", seqs[:3])

20 sequences, e.g. [[50, 20, 10, 25, 60, 70], [50, 20, 10, 60, 25, 70], [50, 20, 10, 60, 70, 25]]


## 4.10 Check Subtree
T1 is huge, T2 is smaller. Is T2 a subtree of T1 (identical structure and values below some node)?
- **Approach 1**: compare pre-order strings with explicit null markers → `O(n + m)` time and space.
- **Approach 2**: for each node in T1 matching T2's root, run a tree-equality check → `O(n + km)` time, `O(log n + log m)` space.

In [14]:
def serialize(n: TreeNode | None) -> str:
    parts: list[str] = []

    def go(x: TreeNode | None) -> None:
        if x is None:
            parts.append("X")
            return
        parts.append(str(x.val))
        go(x.left)
        go(x.right)

    go(n)
    return "," + ",".join(parts) + ","  # delimiters prevent "2" matching "12"


def is_subtree_string(t1: TreeNode | None, t2: TreeNode | None) -> bool:
    return serialize(t2) in serialize(t1)


def same_tree(a: TreeNode | None, b: TreeNode | None) -> bool:
    if a is None or b is None:
        return a is b
    return a.val == b.val and same_tree(a.left, b.left) and same_tree(a.right, b.right)


def is_subtree(t1: TreeNode | None, t2: TreeNode | None) -> bool:
    if t2 is None:
        return True
    if t1 is None:
        return False
    return same_tree(t1, t2) or is_subtree(t1.left, t2) or is_subtree(t1.right, t2)


big = build_bst([50, 20, 60, 10, 25, 70, 5, 15, 65, 80])
small = build_bst([70, 65, 80])
for fn in (is_subtree, is_subtree_string):
    assert fn(big, small)
    assert not fn(big, build_bst([70, 65]))
print("4.10 ok")

4.10 ok


## 4.11 Random Node
A binary tree class with insert/find/delete plus `get_random_node()` returning each node with equal probability. Store subtree sizes: pick index `i` in `[0, size)` and descend using the left subtree size — `O(log N)` on a balanced tree.

In [15]:
class SizedNode:
    def __init__(self, val: int) -> None:
        self.val = val
        self.left: SizedNode | None = None
        self.right: SizedNode | None = None
        self.size = 1

    def insert(self, v: int) -> None:
        if v <= self.val:
            if self.left: self.left.insert(v)
            else: self.left = SizedNode(v)
        else:
            if self.right: self.right.insert(v)
            else: self.right = SizedNode(v)
        self.size += 1

    def find(self, v: int) -> SizedNode | None:
        if v == self.val:
            return self
        child = self.left if v < self.val else self.right
        return child.find(v) if child else None

    def get_ith(self, i: int) -> SizedNode:
        left_size = self.left.size if self.left else 0
        if i < left_size:
            return self.left.get_ith(i)  # type: ignore[union-attr]
        if i == left_size:
            return self
        return self.right.get_ith(i - left_size - 1)  # type: ignore[union-attr]

    def get_random_node(self) -> SizedNode:
        return self.get_ith(random.randrange(self.size))


rt = SizedNode(50)
for v in [30, 70, 20, 40, 60, 80]:
    rt.insert(v)
from collections import Counter
hits = Counter(rt.get_random_node().val for _ in range(70_000))
assert set(hits) == {20, 30, 40, 50, 60, 70, 80}
assert all(8_500 < c < 11_500 for c in hits.values()), hits
print("4.11 ok", dict(sorted(hits.items())))

4.11 ok {20: 9970, 30: 9888, 40: 10089, 50: 10020, 60: 10011, 70: 9889, 80: 10133}


## 4.12 Paths with Sum
Count downward paths (any start, any end, moving parent→child) whose values sum to a target. Brute force is `O(N log N)` on a balanced tree; the prefix-sum hash map gives `O(N)`: at each node, the number of paths ending here equals the count of earlier prefix sums equal to `running - target`.

In [16]:
def count_paths_with_sum(root: TreeNode | None, target: int) -> int:
    prefix_counts: defaultdict[int, int] = defaultdict(int)
    prefix_counts[0] = 1

    def go(n: TreeNode | None, running: int) -> int:
        if n is None:
            return 0
        running += n.val
        total = prefix_counts[running - target]
        prefix_counts[running] += 1
        total += go(n.left, running) + go(n.right, running)
        prefix_counts[running] -= 1  # backtrack
        return total

    return go(root, 0)


def count_paths_brute(root: TreeNode | None, target: int) -> int:
    def from_node(n: TreeNode | None, s: int) -> int:
        if n is None:
            return 0
        s += n.val
        return (s == target) + from_node(n.left, s) + from_node(n.right, s)

    if root is None:
        return 0
    return from_node(root, 0) + count_paths_brute(root.left, target) + count_paths_brute(root.right, target)


#        10
#      5    -3
#    3   2     11
#   3 -2  1
tree = TreeNode(10,
                TreeNode(5, TreeNode(3, TreeNode(3), TreeNode(-2)), TreeNode(2, None, TreeNode(1))),
                TreeNode(-3, None, TreeNode(11)))
assert count_paths_with_sum(tree, 8) == count_paths_brute(tree, 8) == 3
for _ in range(50):
    rand_tree = build_bst(random.sample(range(-10, 10), 12))
    tgt = random.randint(-10, 10)
    assert count_paths_with_sum(rand_tree, tgt) == count_paths_brute(rand_tree, tgt)
print("4.12 ok")

4.12 ok
